In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/processed/merged.csv", index_col="time", parse_dates=True)

# Restore the hours we dropped, as empty rows, so the timeline has no jumps
data = data.asfreq("h")

print("Rows:", len(data))
print("From", data.index.min(), "to", data.index.max())

Rows: 17544
From 2024-01-01 00:00:00 to 2025-12-31 23:00:00


In [2]:
data["target_pm25_24h"] = data["pm25"].shift(-24)
data[["pm25", "target_pm25_24h"]].head(26)

,pm25,target_pm25_24h
time,,
2024-01-01 00:00:00,192.075000,204.950000
2024-01-01 01:00:00,170.900000,172.425000
2024-01-01 02:00:00,155.450000,181.400000
2024-01-01 03:00:00,187.275000,179.300000
2024-01-01 04:00:00,127.575000,196.825000
2024-01-01 05:00:00,106.600000,214.075000
2024-01-01 06:00:00,115.925000,239.275000
2024-01-01 07:00:00,120.925000,232.675000
2024-01-01 08:00:00,121.000000,238.100000


In [4]:
data["hour"] = data.index.hour
data["month"] = data.index.month
data["dayofweek"] = data.index.day_of_week  # 0 = Monday

# Clock-style encoding, so 23:00 and 00:00 count as neighbours
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)
data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

# Delhi's smog season
data["is_winter"] = data["month"].isin([10, 11, 12, 1]).astype(int)

In [5]:
rad = np.deg2rad(data["wind_dir"])
data["wind_u"] = data["wind_speed"] * np.sin(rad)  # east-west
data["wind_v"] = data["wind_speed"] * np.cos(rad)  # north-south
data = data.drop(columns=["wind_dir"])

In [6]:
for lag in [1, 2, 3, 6, 12, 24, 48]:
    data[f"pm25_lag_{lag}h"] = data["pm25"].shift(lag)

data["pm25_mean_6h"] = data["pm25"].rolling(6, min_periods=4).mean()
data["pm25_mean_24h"] = data["pm25"].rolling(24, min_periods=18).mean()
data["pm25_std_24h"] = data["pm25"].rolling(24, min_periods=18).std()

# Change in the last 24 hours: is pollution rising or falling?
data["pm25_change_24h"] = data["pm25"] - data["pm25_lag_24h"]

In [7]:
# Every column except blh must be complete; blh is handled in the experiment later
cols_required = [c for c in data.columns if c != "blh"]

before = len(data)
features = data.dropna(subset=cols_required)
print(f"Rows: {before} → {len(features)}")
print("Columns:", features.shape[1])

features.to_csv("../data/processed/features.csv")
print("Saved!")

Rows: 17544 → 15304
Columns: 41
Saved!


In [8]:
corr = features.drop(columns=["blh"]).corr()["target_pm25_24h"].drop("target_pm25_24h")
corr.sort_values(key=abs, ascending=False).head(15).round(2)

pm25             0.70
pm25_lag_1h      0.67
pm25_mean_6h     0.66
pm25_mean_24h    0.64
pm25_lag_24h     0.64
pm25_lag_2h      0.63
pm10             0.62
pm25_lag_48h     0.60
pm25_lag_3h      0.58
pm25_lag_6h      0.49
pm25_lag_12h     0.49
dew_point       -0.42
is_winter        0.42
pm25_std_24h     0.41
pressure         0.39
Name: target_pm25_24h, dtype: float64